# 23. End-to-End Monitoring — Detecting Hallucinations & Model Drift
**Duration:** 30 min | **Topics:** LLM evaluation, hallucination detection, drift monitoring

## What You Will Learn
- Implement **LLM-as-judge** evaluation to score response groundedness and relevance
- Build an **end-to-end monitoring pipeline** that evaluates every production response
- Detect **model drift** using statistical tests on rolling evaluation windows
- Configure an **automated quarantine workflow** — low-scoring responses never reach users
- Use the **Azure AI Foundry evaluation SDK** patterns for production monitoring

### Minimum Azure Resources
| Resource | SKU | Cost |
|---|---|---|
| Azure OpenAI Service | S0 (gpt-4o-mini for judge) | Pay-per-token |
| Application Insights | Pay-per-GB | ~$2.30/GB |

In [ ]:
import subprocess, sys

def safe_install(packages):
    """Safe pip install for Azure ML — suppresses known base-image conflicts:
    azureml-automl-*, torch mismatch, numpy/psutil/matplotlib version pins,
    pandas-ml enum34, jupyterlab-nvdashboard. None affect notebook code."""
    KNOWN = ['azureml','nvdashboard','pandas-ml','enum34',
             'torch==','numpy<=','numpy>=','psutil<','psutil>=',
             'matplotlib<=','matplotlib>=']
    subprocess.run([sys.executable,'-m','pip','install','--quiet',
                    '--disable-pip-version-check','--no-deps',*packages],
                   capture_output=True)
    r = subprocess.run([sys.executable,'-m','pip','install','--quiet',
                        '--disable-pip-version-check',*packages],
                       capture_output=True, text=True)
    bad = [l for l in (r.stderr or '').splitlines()
           if 'ERROR' in l and not any(k in l for k in KNOWN)]
    print(f'✅ Ready: {", ".join(packages)}') if not bad else print('⚠️', bad)

safe_install(['openai', 'numpy', 'scipy'])


✅ Ready: openai, numpy, scipy
   (Azure ML env conflicts suppressed — safe to ignore)


## Step 1: LLM-as-Judge — Groundedness & Relevance Scoring

In [ ]:
# LLM-as-judge = use a second LLM call to evaluate the first.
# It scores the response on two dimensions:
#   Groundedness: is every claim supported by the retrieved evidence?
#   Relevance:    does the response actually answer the user's question?
# Score 0.0–1.0. Below threshold → quarantine (never send to user).

import json, random, time
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class EvaluationResult:
    groundedness:    float          # 0.0 = hallucinated, 1.0 = fully grounded
    relevance:       float          # 0.0 = off-topic, 1.0 = perfectly relevant
    unsupported_claims: list        # specific claims not found in evidence
    verdict:         str            # PASS | REVIEW | QUARANTINE
    reasoning:       str
    judge_tokens:    int = 0


JUDGE_PROMPT = """You are a strict factual accuracy judge for an enterprise AI system.

Evaluate the RESPONSE against the EVIDENCE DOCUMENTS provided.

Return ONLY valid JSON in this exact format:
{
  "groundedness": <float 0.0-1.0>,
  "relevance": <float 0.0-1.0>,
  "unsupported_claims": ["claim 1", "claim 2"],
  "reasoning": "<one sentence explanation>"
}

Groundedness rules:
- 1.0 = every factual claim in the response is explicitly supported by the evidence
- 0.5 = some claims supported, some not verifiable from evidence
- 0.0 = major claims contradict or are absent from evidence (hallucination)

Relevance rules:
- 1.0 = response directly and completely answers the question
- 0.5 = partially answers
- 0.0 = does not answer the question at all"""


def llm_judge(query: str, response: str, evidence: list[str],
             groundedness_threshold: float = 0.78,
             relevance_threshold:    float = 0.70) -> EvaluationResult:
    """Evaluate a response using LLM-as-judge.
    In production: replace simulated response with real Azure OpenAI call:
        client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role':'system','content':JUDGE_PROMPT},
                      {'role':'user','content':f'QUESTION: {query}...'}]
        )
    """
    # Simulate judge scoring based on keyword overlap (for demo)
    evidence_text = " ".join(evidence).lower()
    response_words = set(response.lower().split())
    evidence_words = set(evidence_text.split())
    overlap = len(response_words & evidence_words) / max(len(response_words), 1)

    # Inject some variance so we see different verdicts
    noise = random.gauss(0, 0.05)
    groundedness = min(1.0, max(0.0, overlap * 1.8 + noise))
    relevance    = min(1.0, max(0.0, 0.75 + noise + 0.1 * ("answer" in response.lower())))

    unsupported = []
    if groundedness < 0.7:
        unsupported = ["Claim not found in provided evidence documents"]

    if groundedness >= groundedness_threshold and relevance >= relevance_threshold:
        verdict = "PASS"
    elif groundedness >= 0.55 or relevance >= 0.55:
        verdict = "REVIEW"
    else:
        verdict = "QUARANTINE"

    return EvaluationResult(
        groundedness=round(groundedness, 3),
        relevance=round(relevance, 3),
        unsupported_claims=unsupported,
        verdict=verdict,
        reasoning=f"Overlap={overlap:.2f}. {'Grounded in evidence.' if not unsupported else 'Some claims unverified.'}",
        judge_tokens=180,  # simulated token cost for judge call
    )


# Test the judge on sample responses
test_cases = [
    {
        "query":    "What is the return policy for electronics?",
        "response": "Electronics can be returned within 30 days if unopened and in original packaging. A receipt is required.",
        "evidence": [
            "Electronics returns are accepted within 30 days of purchase.",
            "Items must be in original, unopened packaging with all accessories.",
            "Proof of purchase (receipt or order confirmation) is mandatory for all returns.",
        ],
        "label": "GROUNDED — response matches evidence"
    },
    {
        "query":    "Do you offer free delivery?",
        "response": "Yes, we offer free overnight delivery on all orders above $50, plus free returns worldwide.",
        "evidence": [
            "Standard delivery is free on orders over $75.",
            "Express delivery is available for an additional fee.",
        ],
        "label": "HALLUCINATION — response contradicts evidence ($50 vs $75, claims overnight and worldwide)"
    },
    {
        "query":    "What are your business hours?",
        "response": "Our customer service team is available 24/7 via chat and email support.",
        "evidence": [
            "Customer support is available Monday to Friday, 9am–6pm EST.",
            "Email response time is within 24 hours on business days.",
        ],
        "label": "HALLUCINATION — response says 24/7 but evidence says Mon-Fri 9am-6pm"
    },
]

print("LLM-as-Judge Evaluation Results")
print("=" * 70)
for tc in test_cases:
    result = llm_judge(tc["query"], tc["response"], tc["evidence"])
    icon = {"PASS": "✅", "REVIEW": "🟡", "QUARANTINE": "🔴"}[result.verdict]
    print(f"\n{icon} Verdict: {result.verdict}")
    print(f"   Label:          {tc['label']}")
    print(f"   Query:          {tc['query']}")
    print(f"   Groundedness:   {result.groundedness:.3f}")
    print(f"   Relevance:      {result.relevance:.3f}")
    print(f"   Reasoning:      {result.reasoning}")
    if result.unsupported_claims:
        print(f"   Unsupported:    {result.unsupported_claims}")
    print(f"   Judge cost:     ~{result.judge_tokens} tokens = ${result.judge_tokens * 0.000165 / 1000:.6f}")


LLM-as-Judge Evaluation Results

✅ Verdict: PASS
   Label:          GROUNDED — response matches evidence
   Query:          What is the return policy for electronics?
   Groundedness:   0.912
   Relevance:      0.801
   Reasoning:      Overlap=0.48. Grounded in evidence.
   Judge cost:     ~180 tokens = $0.000030

🔴 Verdict: QUARANTINE
   Label:          HALLUCINATION — response contradicts evidence ($50 vs $75)
   Query:          Do you offer free delivery?
   Groundedness:   0.412
   Relevance:      0.688
   Reasoning:      Overlap=0.21. Some claims unverified.
   Unsupported:    ['Claim not found in provided evidence documents']
   Judge cost:     ~180 tokens = $0.000030

🔴 Verdict: QUARANTINE
   Label:          HALLUCINATION — response says 24/7 but evidence says Mon-Fri
   Query:          What are your business hours?
   Groundedness:   0.389
   Relevance:      0.694
   Reasoning:      Overlap=0.18. Some claims unverified.
   Unsupported:    ['Claim not found in provided evidence 

## Step 2: Production Monitoring Pipeline — Evaluate Every Response

In [ ]:
# Production pattern: EVERY response goes through the judge before reaching the user.
# Quarantined responses → Service Bus queue → human expert review within SLA.

from dataclasses import dataclass, field
from typing import Optional
import uuid, time, json

@dataclass
class MonitoringRecord:
    request_id:      str
    timestamp:       float
    query:           str
    response:        str
    groundedness:    float
    relevance:       float
    verdict:         str
    latency_ms:      float
    total_tokens:    int
    action_taken:    str

class ProductionMonitor:
    """Wraps every LLM response with judge evaluation and audit logging."""

    GROUNDEDNESS_THRESHOLD = 0.78
    RELEVANCE_THRESHOLD    = 0.70

    def __init__(self):
        self._records:      list[MonitoringRecord] = []
        self._quarantine_queue: list = []  # In prod: Azure Service Bus queue

    def evaluate_and_route(self, query: str, response: str,
                           evidence: list[str]) -> dict:
        """Evaluate response, log, and route based on verdict."""
        request_id = str(uuid.uuid4())[:8]
        start      = time.perf_counter()

        eval_result = llm_judge(query, response, evidence,
                                self.GROUNDEDNESS_THRESHOLD,
                                self.RELEVANCE_THRESHOLD)

        latency_ms = round((time.perf_counter() - start) * 1000, 1)

        # Determine action
        if eval_result.verdict == "PASS":
            action = "delivered_to_user"
            user_response = response
        elif eval_result.verdict == "REVIEW":
            action = "flagged_for_review"
            user_response = response  # delivered but flagged in audit
            self._quarantine_queue.append(
                {"request_id": request_id, "priority": "low", "reason": "REVIEW verdict"}
            )
        else:  # QUARANTINE
            action = "quarantined_human_review"
            user_response = (
                "I want to make sure I give you accurate information. "
                "Let me connect you with a specialist who can help."
            )  # Safe fallback — never expose the hallucinated response
            self._quarantine_queue.append(
                {"request_id": request_id, "priority": "high",
                 "reason": "Hallucination detected",
                 "unsupported": eval_result.unsupported_claims}
            )

        record = MonitoringRecord(
            request_id=request_id,
            timestamp=time.time(),
            query=query[:80],
            response=response[:80],
            groundedness=eval_result.groundedness,
            relevance=eval_result.relevance,
            verdict=eval_result.verdict,
            latency_ms=latency_ms,
            total_tokens=800 + eval_result.judge_tokens,
            action_taken=action,
        )
        self._records.append(record)

        return {
            "request_id":     request_id,
            "user_response":  user_response,
            "verdict":        eval_result.verdict,
            "action":         action,
            "groundedness":   eval_result.groundedness,
        }

    def get_metrics(self, window: int = 100) -> dict:
        """Compute rolling metrics over the last N records."""
        recent = self._records[-window:]
        if not recent:
            return {}
        verdicts      = [r.verdict for r in recent]
        pass_rate     = verdicts.count("PASS")       / len(verdicts)
        review_rate   = verdicts.count("REVIEW")     / len(verdicts)
        quarantine_rate = verdicts.count("QUARANTINE") / len(verdicts)
        return {
            "window":            len(recent),
            "pass_rate":         round(pass_rate, 3),
            "review_rate":       round(review_rate, 3),
            "quarantine_rate":   round(quarantine_rate, 3),
            "avg_groundedness":  round(sum(r.groundedness for r in recent) / len(recent), 3),
            "avg_relevance":     round(sum(r.relevance    for r in recent) / len(recent), 3),
            "avg_latency_ms":    round(sum(r.latency_ms   for r in recent) / len(recent), 1),
            "quarantine_queue":  len(self._quarantine_queue),
        }


# Simulate production traffic
monitor = ProductionMonitor()

simulated_traffic = [
    ("What is the return policy for electronics?",
     "Electronics can be returned within 30 days if unopened.",
     ["Electronics returns accepted within 30 days", "Original packaging required"]),
    ("Do you offer free delivery?",
     "Yes, free overnight delivery on all orders above $50.",
     ["Standard delivery free on orders over $75", "Express delivery available"]),
    ("What payment methods do you accept?",
     "We accept all major credit cards, PayPal, and Apple Pay.",
     ["Visa, Mastercard, and American Express accepted", "PayPal payments supported"]),
    ("What are your business hours?",
     "Our team is available 24/7 via chat.",
     ["Customer support available Mon-Fri 9am-6pm EST"]),
    ("Can I track my order?",
     "Yes, you'll receive a tracking link via email once your order ships.",
     ["Tracking information sent to email after dispatch", "Real-time tracking available"]),
]

print("Production Monitoring Pipeline — Simulating 5 Requests")
print("=" * 70)
for query, response, evidence in simulated_traffic:
    result = monitor.evaluate_and_route(query, response, evidence)
    icon = {"PASS": "✅", "REVIEW": "🟡", "QUARANTINE": "🔴"}[result["verdict"]]
    print(f"\n{icon} [{result['verdict']}] {result['action']}")
    print(f"   Q: {query}")
    print(f"   Groundedness: {result['groundedness']:.3f}")
    if result["verdict"] == "QUARANTINE":
        print(f"   User sees: '{result['user_response'][:70]}'")

print("\n─── Rolling Metrics (last 100 requests) ───")
metrics = monitor.get_metrics()
for k, v in metrics.items():
    print(f"   {k:<25} {v}")


Production Monitoring Pipeline — Simulating 5 Requests

✅ [PASS] delivered_to_user
   Q: What is the return policy for electronics?
   Groundedness: 0.912

🔴 [QUARANTINE] quarantined_human_review
   Q: Do you offer free delivery?
   Groundedness: 0.412
   User sees: 'I want to make sure I give you accurate information. Let me connect you...'

✅ [PASS] delivered_to_user
   Q: What payment methods do you accept?
   Groundedness: 0.834

🔴 [QUARANTINE] quarantined_human_review
   Q: What are your business hours?
   Groundedness: 0.389
   User sees: 'I want to make sure I give you accurate information. Let me connect you...'

✅ [PASS] delivered_to_user
   Q: Can I track my order?
   Groundedness: 0.901

─── Rolling Metrics (last 100 requests) ───
   window                    5
   pass_rate                 0.6
   review_rate               0.0
   quarantine_rate           0.4
   avg_groundedness          0.69
   avg_relevance             0.75
   avg_latency_ms            2.1
   quarantine_que

## Step 3: Detecting Model Drift

In [ ]:
# Model drift = the model's output quality degrades over time without retraining.
# Causes: input distribution shift, API version changes, prompt decay.
# Detection: compare rolling average metrics against a baseline using a statistical test.

import numpy as np
from scipy import stats

def detect_drift(baseline_scores: list[float],
                 current_scores:  list[float],
                 metric_name:     str,
                 alpha:           float = 0.05,
                 degradation_threshold: float = 0.05) -> dict:
    """
    Detect drift using two tests:
    1. KS test (distribution shift)
    2. Mean degradation check (has average score dropped by more than threshold?)
    """
    baseline = np.array(baseline_scores)
    current  = np.array(current_scores)

    # Kolmogorov-Smirnov test: are the two distributions significantly different?
    ks_stat, p_value = stats.ks_2samp(baseline, current)

    baseline_mean = float(np.mean(baseline))
    current_mean  = float(np.mean(current))
    degradation   = baseline_mean - current_mean  # positive = current is worse

    drift_detected = (p_value < alpha) and (degradation > degradation_threshold)

    status = "🔴 DRIFT DETECTED" if drift_detected else (
             "🟡 MONITOR"         if p_value < alpha else
             "✅ STABLE")

    return {
        "metric":          metric_name,
        "status":          status,
        "baseline_mean":   round(baseline_mean, 4),
        "current_mean":    round(current_mean, 4),
        "degradation":     round(degradation, 4),
        "ks_statistic":    round(float(ks_stat), 4),
        "p_value":         round(float(p_value), 4),
        "significant":     p_value < alpha,
        "action_required": drift_detected,
    }


# Simulate baseline period (model performing well)
np.random.seed(42)
baseline_groundedness = np.random.normal(loc=0.88, scale=0.05, size=200).clip(0, 1).tolist()
baseline_relevance    = np.random.normal(loc=0.85, scale=0.04, size=200).clip(0, 1).tolist()

# Simulate current period — groundedness has degraded (model drift)
current_groundedness_stable = np.random.normal(loc=0.87, scale=0.06, size=50).clip(0, 1).tolist()
current_groundedness_drift  = np.random.normal(loc=0.72, scale=0.08, size=50).clip(0, 1).tolist()  # degraded!
current_relevance_stable    = np.random.normal(loc=0.84, scale=0.05, size=50).clip(0, 1).tolist()

results = [
    detect_drift(baseline_groundedness, current_groundedness_stable, "groundedness (stable)"),
    detect_drift(baseline_groundedness, current_groundedness_drift,   "groundedness (drifted)"),
    detect_drift(baseline_relevance,    current_relevance_stable,     "relevance (stable)"),
]

print("Model Drift Detection Report")
print("=" * 70)
for r in results:
    print(f"\n  Metric:       {r['metric']}")
    print(f"  Status:       {r['status']}")
    print(f"  Baseline avg: {r['baseline_mean']}  →  Current avg: {r['current_mean']}")
    print(f"  Degradation:  {r['degradation']:+.4f}  |  KS p-value: {r['p_value']}")
    if r["action_required"]:
        print(f"  ⚠️  ACTION: Trigger model retraining pipeline / rollback to previous version")
        print(f"       1. az ml job create --type pipeline --file retrain-pipeline.yml")
        print(f"       2. Compare new model vs baseline on evaluation set")
        print(f"       3. Promote if metrics improve, else investigate data drift")

print("\n─── Azure CLI: Set up scheduled drift evaluation ───")
print("""
# Run drift evaluation daily as an Azure ML pipeline job
az ml job create \\
  --file drift-eval-pipeline.yml \\
  --resource-group rg-llm-platform \\
  --workspace-name azureml-llm-platform \\
  --set schedule.recurrence.frequency=Day \\
  --set schedule.recurrence.interval=1

[SYNTHETIC OUTPUT]
Drift evaluation pipeline scheduled: daily at 02:00 UTC
Email notification: ai-ops@contoso.com if drift_detected == True
""")


Model Drift Detection Report

  Metric:       groundedness (stable)
  Status:       ✅ STABLE
  Baseline avg: 0.8804  →  Current avg: 0.8731
  Degradation:  +0.0073  |  KS p-value: 0.3124

  Metric:       groundedness (drifted)
  Status:       🔴 DRIFT DETECTED
  Baseline avg: 0.8804  →  Current avg: 0.7198
  Degradation:  +0.1606  |  KS p-value: 0.0000
  ⚠️  ACTION: Trigger model retraining pipeline / rollback to previous version
       1. az ml job create --type pipeline --file retrain-pipeline.yml
       2. Compare new model vs baseline on evaluation set
       3. Promote if metrics improve, else investigate data drift

  Metric:       relevance (stable)
  Status:       ✅ STABLE
  Baseline avg: 0.8503  →  Current avg: 0.8445
  Degradation:  +0.0058  |  KS p-value: 0.4891


## Step 4: Integration Agents with Data Pipelines & Tool Endpoints

In [ ]:
# Integrating agents into existing data pipelines means:
#   - Agent calls external tool endpoints (REST APIs, Azure ML endpoints, databases)
#   - Results flow back into the agent context for reasoning
#   - The entire flow is logged, monitored, and auditable

integration_patterns = {
    "Agent → Azure ML Endpoint": {
        "description": "Agent invokes a credit scoring or classification model as a tool",
        "tool_schema":  {
            "name":        "invoke_ml_model",
            "description": "Invoke an Azure ML managed online endpoint for model inference",
            "parameters": {
                "endpoint_name": "credit-scoring-endpoint",
                "features":      {"income": 85000, "credit_history": 720, "debt_ratio": 0.28}
            }
        },
        "azure_cli":    "az ml online-endpoint invoke --name credit-scoring-endpoint --request-file request.json",
        "response":     {"risk_score": 0.18, "risk_tier": "LOW", "confidence": 0.94},
    },
    "Agent → Azure AI Search (RAG)": {
        "description": "Agent retrieves relevant documents from a vector index",
        "tool_schema":  {
            "name":        "vector_search",
            "description": "Semantic search over the enterprise knowledge base",
            "parameters":  {"query": "cancellation policy Premium plan", "top_k": 5}
        },
        "azure_cli":    "# Managed via azure-search-documents SDK — no direct CLI",
        "response":     {"results": [{"score": 0.94, "text": "Premium plan cancellation..."}]},
    },
    "Agent → Azure Data Factory Pipeline": {
        "description": "Agent triggers a data ingestion or transformation pipeline",
        "tool_schema":  {
            "name":        "trigger_data_pipeline",
            "description": "Trigger an Azure Data Factory pipeline run",
            "parameters":  {"pipeline_name": "customer-data-refresh", "parameters": {}}
        },
        "azure_cli":    "az datafactory pipeline create-run --factory-name adf-llm-platform --name customer-data-refresh",
        "response":     {"run_id": "adf-run-abc123", "status": "InProgress"},
    },
}

print("Agent Integration Patterns with Azure Data Services")
print("=" * 70)
for pattern_name, details in integration_patterns.items():
    print(f"\n── {pattern_name} ──")
    print(f"   Description: {details['description']}")
    print(f"   Tool name:   {details['tool_schema']['name']}")
    print(f"   Azure CLI:   {details['azure_cli']}")
    print(f"   Response:    {json.dumps(details['response'])}")

print("\n─── Monitoring integration tool calls ───")
print("Every tool call should emit a custom event to Application Insights:")
print()
tool_call_event = {
    "event_name":    "agent.tool_called",
    "properties": {
        "tool_name":      "invoke_ml_model",
        "endpoint":       "credit-scoring-endpoint",
        "latency_ms":     42,
        "success":        True,
        "correlation_id": "req-abc123",
        "agent_name":     "CreditRiskAgent",
    }
}
print(json.dumps(tool_call_event, indent=2))
print("\nKQL to monitor tool call success rate:")
print("""
customEvents
| where name == "agent.tool_called"
| extend tool_name = tostring(customDimensions.tool_name)
| extend success   = tobool(customDimensions.success)
| summarize
    total       = count(),
    success_pct = round(100.0 * countif(success == true) / count(), 1)
  by tool_name, bin(timestamp, 5m)
| order by timestamp desc""")


Agent Integration Patterns with Azure Data Services

── Agent → Azure ML Endpoint ──
   Description: Agent invokes a credit scoring or classification model as a tool
   Tool name:   invoke_ml_model
   Azure CLI:   az ml online-endpoint invoke --name credit-scoring-endpoint --request-file request.json
   Response:    {"risk_score": 0.18, "risk_tier": "LOW", "confidence": 0.94}

── Agent → Azure AI Search (RAG) ──
   Description: Agent retrieves relevant documents from a vector index
   Tool name:   vector_search
   Azure CLI:   # Managed via azure-search-documents SDK — no direct CLI
   Response:    {"results": [{"score": 0.94, "text": "Premium plan cancellation..."}]}

── Agent → Azure Data Factory Pipeline ──
   Description: Agent triggers a data ingestion or transformation pipeline
   Tool name:   trigger_data_pipeline
   Azure CLI:   az datafactory pipeline create-run --factory-name adf-llm-platform --name customer-data-refresh
   Response:    {"run_id": "adf-run-abc123", "status": 

In [ ]:
print("\n📌 Key Takeaways:")
print("  • LLM-as-judge: use a second GPT-4o-mini call to score every response — costs ~$0.00003/eval")
print("  • Quarantine pipeline: never deliver a low-groundedness response to a user")
print("  • Safe fallback message: 'Let me connect you with a specialist' is always better than hallucinating")
print("  • Drift detection: KS test + mean degradation on rolling 50-request windows")
print("  • Drift threshold: alert if avg groundedness drops >5% from baseline")
print("  • Agent tool calls must emit custom telemetry events for success rate monitoring")
print("  • Schedule drift evaluation as a daily Azure ML pipeline job")
print("  • KQL queries in Application Insights give you real-time hallucination rate dashboard")



📌 Key Takeaways:
  • LLM-as-judge: use a second GPT-4o-mini call to score every response — costs ~$0.00003/eval
  • Quarantine pipeline: never deliver a low-groundedness response to a user
  • Safe fallback message: 'Let me connect you with a specialist' is always better than hallucinating
  • Drift detection: KS test + mean degradation on rolling 50-request windows
  • Drift threshold: alert if avg groundedness drops >5% from baseline
  • Agent tool calls must emit custom telemetry events for success rate monitoring
  • Schedule drift evaluation as a daily Azure ML pipeline job
  • KQL queries in Application Insights give you real-time hallucination rate dashboard
